# H11 — 重启之后还记得设置

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H11 — 重启之后还记得设置。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H11-storage.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

工具重启就丢失名字、亮度和采样周期。保存配置还要面对无效值、升级、磨损与恢复，而不只会 set/get。

前置：H10；已记录配置和 Flash 数据。

## 本次够用的知识

NVS 按 namespace/key 保存类型化值，set 后 commit 才持久化。不存在的 key 与读错误不同；读失败不能悄悄清空全设备。配置版本描述数据格式变化，不等于固件版本。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 定义配置契约 · Guided

选三项配置：device_name、brightness、sample_ms。写出默认值、类型、范围与重启预期；不要频繁保存传感器 raw。
保留 H10 的状态 owner；持久化由配置模块处理，网络输入通过同一校验路径。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 先验证 commit 和重启 · Guided

REQUIRES nvs_flash freertos log。示例只在启动写一次 boots counter，观察 Reset 后递增，区分 flash monitor 与 erase-flash。
已有 H07 初始化 NVS，合并时不能每个模块重新初始化/erase；将示例读写逻辑转为函数，复用初始化结果。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H11/examples/nvs_config.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES nvs_flash freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "nvs.h"
#include "nvs_flash.h"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    ESP_ERROR_CHECK(
        nvs_flash_init()); // Stop and diagnose; never silently erase all namespaces.
    nvs_handle_t handle;
    ESP_ERROR_CHECK(nvs_open("desk", NVS_READWRITE, &handle));
    uint32_t boots = 0;
    const esp_err_t result = nvs_get_u32(handle, "boots", &boots);
    if (result != ESP_OK && result != ESP_ERR_NVS_NOT_FOUND) {
        nvs_close(handle);
        ESP_ERROR_CHECK(result);
        return;
    }
    if (boots == UINT32_MAX) {
        ESP_LOGE("nvs", "counter full; reset explicitly instead of wrapping");
        nvs_close(handle);
        return;
    }
    ++boots;
    ESP_ERROR_CHECK(nvs_set_u32(handle, "boots", boots));
    ESP_ERROR_CHECK(nvs_commit(handle));
    nvs_close(handle);
    ESP_LOGI("nvs", "boots=%lu", (unsigned long)boots);
    while (true) {
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


## 少一次 commit 会怎样 · Modify

暂时去掉 nvs_commit，重启前后记录 boots；恢复。set 返回 OK 不代表重启后有值。
将每 tick 写改为“值变化后延迟合并保存”，定义掉电前尚未保存的窗口；配置成功 UI 要区分 accepted 和 persisted。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 类型变化与缺字段 · Debug

在测试 namespace 将 key 从 u32 改为 string，再用旧读取 API，记录类型错误；不要破坏实际 Wi-Fi namespace。
故意给 sample_ms=0 或超范围值，验证拒绝且旧配置保留。遇 NVS_NO_FREE_PAGES / NEW_VERSION_FOUND 先诊断与备份，不默认 erase 整块。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 版本迁移与恢复 · Build

实现 schema 1→2：增加 brightness 字段，保留旧有效 name/sample_ms。不存在时默认，非法值拒绝或明确回退；迁移写成功后再标版本。
设计可恢复写入（双槽 blob+version 或明确支持的提交策略），不要假定多个 key 的 commit 等于应用级跨字段原子事务。恢复出厂仅清理本应用 namespace，网络凭据另行授权。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 一张配置故障表 · Open

测试初次启动、重启、非法类型、旧版本、未来版本、写失败与掉电模拟；host 注入失败点，实机复位与恢复分开记录。
H12 配置来源会从电脑变成手机，校验/存储模块仍复用。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] commit 后重启保留，未 commit 情况有记录
- [ ] 无效配置不会破坏旧值，未来 schema 有策略
- [ ] 出厂恢复范围与写失败恢复明确

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H11',
    [
        'commit 后重启保留，未 commit 情况有记录',
        '无效配置不会破坏旧值，未来 schema 有策略',
        '出厂恢复范围与写失败恢复明确',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H11/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/storage/nvs_flash.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/storage/nvs_flash.html)